In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Title:
Background Noise & Focus Dataset

## Description:
The Background Noise & Focus Dataset is designed to analyze how different types of ambient sounds influence human concentration, productivity, and cognitive performance. This dataset includes categorized audio samples such as white noise, café ambience, office chatter, nature sounds, traffic noise, instrumental music, and silence-based controls.

Each record is structured with features including noise type, decibel level, duration of exposure, task type (e.g., reading, coding, problem-solving), participant response time, accuracy score, and self-reported focus rating. Additional metadata such as age group, work environment, and time of day may also be included for deeper behavioral insights.

## Import dataset

In [ ]:
df = pd.read_csv("/kaggle/input/background-noise-and-focus-dataset/background_noise_focus_dataset.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.columns

## Feature engg

In [ ]:
plt.figure()
sns.histplot(df['age'], bins=15, kde=True)
plt.title("Age Distribution of Participants")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()


In [ ]:
plt.figure()
sns.countplot(x='role', data=df)
plt.title("Participant Role Distribution")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure()
sns.countplot(x='background_noise_type', data=df)
plt.title("Background Noise Type Distribution")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure()
sns.barplot(x='background_noise_type',
            y='focus_duration_minutes',
            data=df,
            estimator='mean')
plt.title("Average Focus Duration by Noise Type")
plt.xticks(rotation=45)
plt.ylabel("Avg Focus Duration (Minutes)")
plt.show()


In [ ]:
plt.figure()
sns.scatterplot(x='noise_volume_level',
                y='perceived_focus_score',
                hue='background_noise_type',
                data=df)
plt.title("Noise Volume vs Perceived Focus Score")
plt.xlabel("Noise Volume Level")
plt.ylabel("Perceived Focus Score")
plt.show()


In [ ]:
plt.figure()
sns.boxplot(x='task_type',
            y='task_completion_quality',
            data=df)
plt.title("Task Completion Quality by Task Type")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure()
sns.boxplot(x='background_noise_type',
            y='mental_fatigue_after_task',
            data=df)
plt.title("Mental Fatigue After Task by Noise Type")
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure()
numeric_cols = df.select_dtypes(include=['int64', 'float64'])

corr_matrix = numeric_cols.corr()

sns.heatmap(corr_matrix,
            annot=True,
            cmap='coolwarm',
            fmt=".2f")

plt.title("Correlation Matrix")
plt.show()


In [ ]:
plt.figure()
sns.scatterplot(x='focus_duration_minutes',
                y='mental_fatigue_after_task',
                hue='task_type',
                data=df)
plt.title("Focus Duration vs Mental Fatigue")
plt.xlabel("Focus Duration (Minutes)")
plt.ylabel("Mental Fatigue After Task")
plt.show()


In [ ]:
pivot_table = df.pivot_table(
    values='perceived_focus_score',
    index='task_type',
    columns='background_noise_type',
    aggfunc='mean'
)

plt.figure()
sns.heatmap(pivot_table,
            annot=True,
            cmap='YlGnBu',
            fmt=".2f")

plt.title("Avg Perceived Focus Score")
plt.show()


In [ ]:
sns.pairplot(df,
             vars=['noise_volume_level',
                   'focus_duration_minutes',
                   'perceived_focus_score',
                   'task_completion_quality',
                   'mental_fatigue_after_task'],
             hue='background_noise_type')
plt.show()


## Prepare data for machine learning

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [ ]:
print("Dataset Shape:", df.shape)
print(df.head())

In [ ]:
df = df.drop(columns=['participant_id'])

# Handle missing values
numeric_cols = df.select_dtypes(include=['int64','float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
X = df.drop(columns=['perceived_focus_score'])
y = df['perceived_focus_score']

categorical_cols = ['role', 'task_type', 'background_noise_type']
numeric_cols = X.drop(columns=categorical_cols).columns

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first'), categorical_cols)
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

In [ ]:
rf_pipeline.fit(X_train, y_train)

y_pred = rf_pipeline.predict(X_test)


In [ ]:
def evaluate_model(y_test, y_pred, title="Model Performance"):
    print("\n", title)
    print("MAE :", mean_absolute_error(y_test, y_pred))
    print("MSE :", mean_squared_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print("R2  :", r2_score(y_test, y_pred))

evaluate_model(y_test, y_pred, "Random Forest Initial Performance")


In [ ]:
cv_scores = cross_val_score(
    rf_pipeline, X, y, cv=5, scoring='r2'
)

print("\nCross Validation R2 Scores:", cv_scores)
print("Average CV R2:", cv_scores.mean())

In [ ]:
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\nBest Parameters:", grid_search.best_params_)

best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test)

evaluate_model(y_test, y_pred_best, "Random Forest After Tuning")


In [ ]:
ohe = best_model.named_steps['preprocessor'].named_transformers_['cat']
encoded_cat_features = ohe.get_feature_names_out(categorical_cols)

all_features = list(numeric_cols) + list(encoded_cat_features)

importances = best_model.named_steps['model'].feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\nTop Important Features:")
print(feature_importance_df.head())


## Thank you..pls upvote!!!